# Imports

In [ ]:
import pandas as pd

# Constants

In [99]:
# Raw Data Path
PATH_RAW = "./data/raw/"
# Palpites
FILE_TIPS = "1 - palpites.xlsx"

# Processed Data Path
PATH_PROCESSED = "./data/processed/"

# ETL

In [ ]:
# Lendo dados brutos de palpites
df_tips_gs = pd.read_excel(PATH_RAW+FILE_TIPS, sheet_name="palpites_fg",engine="openpyxl")

# Removendo colunas desnecessárias
# Deixando somente o nome e os palpites
df_tips_gs_2 = df_tips_gs.drop(columns=["Carimbo de data/hora","Deixe uma foto sua aqui", "Campeão", "Vice", "Artilheiro"])

In [88]:
# Derrete as colunas em linhas
# col jogo (nome temporario) recebe o confronto
# gols (nome temporario) recebe os valores 
df_tips_gs_pivot = (
    df_tips_gs_2.melt(id_vars=["Nome"], var_name="col_jogo", value_name="gols")
      .dropna(subset=["gols"])
)

In [89]:
# Quebra o texto que está no padrão forms
# Time A x Time B [Time A]
# Vira :
# nm_cfr = Time A x Time B
# nm_time_palpite = Time A
df_tips_gs_pivot["nm_cfr"] = df_tips_gs_pivot["col_jogo"].str.extract(r"^(.*) \[")[0].str.strip()
df_tips_gs_pivot["nm_time_palpite"] = df_tips_gs_pivot["col_jogo"].str.extract(r"\[(.*)\]")[0].str.strip()
# nm_time_casa = Time A
# nm_time_fora = Time B
parts = df_tips_gs_pivot["col_jogo"].str.extract(r"^(.*?) x (.*?) \[")
df_tips_gs_pivot["nm_time_casa"] = parts[0].str.strip()
df_tips_gs_pivot["nm_time_fora"] = parts[1].str.strip()

In [91]:
# Cria um label para definir se o valor vai pra casa ou fora
df_tips_gs_pivot["side"] = df_tips_gs_pivot.apply(
    lambda r: "vl_casa" if r["nm_time_palpite"] == str(r["nm_time_casa"]) else "vl_fora",
    axis=1,
)
# Unpivota de novo agora para preencher as colunas de vl_fora e vl_casa
df_tips_gs_pivot_unpivot = (
    df_tips_gs_pivot.pivot_table(
        index=["Nome","nm_cfr","nm_time_casa","nm_time_fora"],
        columns="side",
        values="gols",
        aggfunc="first",
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

In [98]:
# Ajustando a ordem e renomeando colunas
df_tips_gs_final = df_tips_gs_pivot_unpivot[['Nome','nm_cfr','nm_time_casa','vl_casa','nm_time_fora', 'vl_fora']]
df_tips_gs_final.rename(columns={'Nome': 'nm_player'})

,nm_player,nm_cfr,nm_time_casa,vl_casa,nm_time_fora,vl_fora
0,ana nath,Alemanha x Costa do Marfim,Alemanha,2,Costa do Marfim,1
1,ana nath,Alemanha x Curaçao,Alemanha,4,Curaçao,2
2,ana nath,Argentina x Argélia,Argentina,1,Argélia,1
3,ana nath,Argentina x Áustria,Argentina,0,Áustria,2
4,ana nath,Argélia x Áustria,Argélia,4,Áustria,3
...,...,...,...,...,...,...
283,washington,Uruguai x Cabo Verde,Uruguai,2,Cabo Verde,4
284,washington,Uruguai x Espanha,Uruguai,2,Espanha,1
285,washington,Uzbequistão x Colômbia,Uzbequistão,2,Colômbia,2
286,washington,África do Sul x Coreia do Sul,África do Sul,4,Coreia do Sul,0


# Save

In [ ]:
# Salvar sem o índice
file_path = PATH_PROCESSED + "palpites_processados.csv"
df_tips_gs_final.to_csv(file_path, index=False, encoding='utf-8')